# Generic real controls for the CM winners

We compare the current CM winners for $D=(1,2)$ and $D=(1,3)$ with non-rational symplectic deformations. Unlike the rational experiment in notebook 08, these transformations are designed to leave the original CM isogeny class.

In [1]:
import sys
from pathlib import Path
from math import sqrt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

from gkp_systole import (
    EisensteinHermitianForm,
    GaussianHermitianForm,
    high_precision_pi_systole,
    scan_pi_symplectic_deformations,
)

## Why these are genuinely generic controls

Each elementary deformation is $S=I+t vv^TA$ with $t=q\pi$ and nonzero rational $q$. It preserves $A$ exactly in the mathematical model but is not a rational transformation, so it does not automatically remain in the starting rational isogeny class. CM points have measure zero in moduli, making generic real controls overwhelmingly non-CM. We do not certify every individual endomorphism ring.

In [2]:
gaussian_12 = GaussianHermitianForm(2, 2, 1, 1)
eisenstein_13 = EisensteinHermitianForm(2, 2, 1, 1)

cases = {
    '(1,2)': dict(A=gaussian_12.alternating, G=gaussian_12.metric, scale=1.0, local_seed=3101, broad_seed=3102),
    '(1,3)': dict(A=eisenstein_13.alternating, G=eisenstein_13.metric_core, scale=1/sqrt(3), local_seed=3201, broad_seed=3202),
}

## 10,000-point scan

For each $D$, we generate 2,000 small local perturbations and 3,000 broader controls. The seeds make the entire experiment reproducible.

In [3]:
scans = {}
for label, case in cases.items():
    local = scan_pi_symplectic_deformations(
        case['A'], case['G'], sample_count=2000, seed=case['local_seed'],
        amplitude=0.02, steps=4, vector_bound=2, metric_scale=case['scale'],
    )
    broad = scan_pi_symplectic_deformations(
        case['A'], case['G'], sample_count=3000, seed=case['broad_seed'],
        amplitude=0.6, steps=4, vector_bound=1, metric_scale=case['scale'],
    )
    scans[label] = (local, broad)
print('Completed', sum(len(scan.samples) for pair in scans.values() for scan in pair), 'generic-real controls')

Completed 10000 generic-real controls


In [4]:
rows = []
for label, (local, broad) in scans.items():
    scale = cases[label]['scale']
    rows.append({
        'D': label,
        'CM baseline ell^2': float(local.baseline_result.squared_systole) * scale,
        'best local ell^2': local.best_sample.squared_systole * scale,
        'best broad ell^2': broad.best_sample.squared_systole * scale,
        'samples beating CM': local.number_beating_baseline + broad.number_beating_baseline,
        'total controls': len(local.samples) + len(broad.samples),
    })
rows

[{'D': '(1,2)',
  'CM baseline ell^2': 1.0,
  'best local ell^2': 0.9765886679992841,
  'best broad ell^2': 0.894080410143457,
  'samples beating CM': 0,
  'total controls': 5000},
 {'D': '(1,3)',
  'CM baseline ell^2': 0.769800358919501,
  'best local ell^2': 0.7544310245220363,
  'best broad ell^2': 0.682880587200975,
  'samples beating CM': 0,
  'total controls': 5000}]

## Independent 60-digit recheck

The four strongest controls are reconstructed from their rational coefficients times $\pi$ and solved again by a separate high-precision Cholesky branch-and-bound routine.

In [5]:
checks = []
for label, (local, broad) in scans.items():
    case = cases[label]
    for regime, scan in (('local', local), ('broad', broad)):
        best = scan.best_sample
        high_precision_core = high_precision_pi_systole(
            case['A'], case['G'], best.parameters, decimal_places=60,
        )
        high_precision_physical = float(high_precision_core) * case['scale']
        screened = best.squared_systole * case['scale']
        checks.append(dict(D=label, regime=regime, screened=screened, high_precision=high_precision_physical))
        assert abs(screened - high_precision_physical) < 1e-12
checks

[{'D': '(1,2)',
  'regime': 'local',
  'screened': 0.9765886679992841,
  'high_precision': 0.9765886679992842},
 {'D': '(1,2)',
  'regime': 'broad',
  'screened': 0.894080410143457,
  'high_precision': 0.8940804101434524},
 {'D': '(1,3)',
  'regime': 'local',
  'screened': 0.7544310245220363,
  'high_precision': 0.7544310245220359},
 {'D': '(1,3)',
  'regime': 'broad',
  'screened': 0.682880587200975,
  'high_precision': 0.682880587200978}]

In [6]:
assert all(scan.number_beating_baseline == 0 for pair in scans.values() for scan in pair)
print('No generic-real control beats its CM baseline: PASS')

No generic-real control beats its CM baseline: PASS


## Interpretation

None of the 10,000 generic-real controls beats the appropriate CM baseline. This is substantially better CM-versus-generic evidence than the rational isogeny-class scan. It remains numerical evidence rather than a proof of global optimality or a point-by-point endomorphism-ring certification.